# N027 · 三数和与双指针消元

**目标：** 将一维单调排除嵌入外层枚举。

**先修：** N025, N020。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 排序；去重；固定元素；双指针；面积上界。

**配套讲解来源：** [同名逐章意见](../../comment/027_k_sum_and_geometric_elimination.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章把 N025 的"相向双指针"装进外层循环，解决两个著名问题。

**问题一（三数和，`three_sum`）**：给你一个整数数组，找出所有"三个数相加等于 0"的组合，答案里不许有重复的三元组。举个具体到数字的例子：输入 `[-1,0,1,2,-1,-4]` → 输出 `[[-1,-1,2],[-1,0,1]]` → 为什么？−1+(−1)+2=0 和 −1+0+1=0 两组成立；虽然数组里有两个 −1、能拼出两组"形如 −1,0,1"的选法（两个 −1 各配一次 0 和 1），但它们数值上相同，只算一组。暴力做法要枚举所有三元组，O(n³)；本章做法是"先排序，再固定一个数，剩下两个数用 N025 的有序两数和（目标 0−固定值）线性解决"，整体 O(n²)。

**问题二（盛水容器，`max_area`）**：给你一排高度 `height[i]`，选两根当容器壁，能装的水量 = 两壁距离 × 两壁中较矮那根的高度，问最多能装多少。例如输入 `[1,8,6,2,5,4,8,3,7]` → 输出 `49` → 为什么？选下标 1（高 8）和下标 8（高 7）两根：距离 7、短板 7，面积 7×7=49，没有任何别的组合更大。它也用相向双指针，但移动指针的理由和三数和完全不同——三数和靠"数值大小"消元，容器靠"几何上界"消元，这是本章想让你对照体会的重点。

## 2. 基础知识：先读懂这些词

- **排序（sorting as preprocessing）**：三数和第一步就把数组排成升序副本。排序花 O(n log n)，换来两个好处：相同值挤到一起（去重跳过才可能写）+ 子数组有序（N025 的单调消元才能用）。
- **固定元素（fixing an element）**：三数和降维的手段。外层 for 逐个"固定"a[i] 当三人组的第一位，剩下的问题变成"在 a[i] 右边找两个数和为 −a[i]"——正是 N025 处理过的有序两数和。三数问题就这样拆成了一个 n 次的线性子问题。
- **双指针消元（two-pointer elimination）**：N025 的老朋友——left/right 两端夹击，和太小扔左、太大扔右。本章把它作为内层循环嵌进外层枚举，"一维单调排除嵌入外层枚举"说的就是这个组合动作。
- **去重跳过（duplicate skipping）**：答案不许重复时，要在三个位置跳过已见过的值：外层固定值重复时跳过整个 i（`a[i]==a[i-1]` 就 continue）；内层命中一组解后，left 要跨过与刚才相同的值，right 镜像地跨过。三处缺一处，答案就会出现重复三元组。
- **面积上界（area bound）**：容器问题的消元依据。面积为"宽度 × 两壁较矮者"。当前配对中较矮的那根壁，如果配上一根**更靠里**的壁：宽度必然变小，高度又不大于当前短板（新配对的高度由较矮者决定，而较矮者不高于当前短板），所以面积不可能超过刚检查过的这个——较矮壁的所有"更近配对"都不值得看，可以放心扔掉。
- **短板效应（min 决定面积）**：水量由较矮的壁决定，高的那根再高也没用。这正是消元时"扔短板、留长板"的直觉来源。
- **提前终止（early break）**：三数和外层的 `if a[i]>0: break`。数组已升序，如果固定的最小数都已大于 0，那么三个正数相加不可能等于 0，后面全不用看。

## 3. 思路推导：从小例子开始

**`three_sum` 的思路：**

- **Step 1：排序。** 例子 `[-1,0,1,2,-1,-4]` 排成 `[-4,-1,-1,0,1,2]`。
- **Step 2：外层固定 a[i]。** i 从 0 走到倒数第三个位置（`range(len(a)-2)`，保证 i 右边至少还有两个数可配）。固定值如果与前一个相同就整轮跳过；如果已大于 0 就提前收工。
- **Step 3：内层双指针找两数和为 −a[i]。** left 从 i+1、right 从末尾出发，和小于 0 扔左，大于 0 扔右，等于 0 收进答案。
- **Step 4：命中后去重跳位。** 记下一组解后 left、right 各进一格，然后各自跨过与前值相同的元素，避免同一个固定值配出重复对。

**手算演示一：`three_sum([-1,0,1,2,-1,-4])`（排序后 `[-4,-1,-1,0,1,2]`）**

| 外层 i | 固定值 | 内层过程 | 结果 |
|---|---|---|---|
| 0 | −4 | left=1,right=5：−4−1+2=−3<0 扔左；−4−1+2=−3<0 扔左；−4+0+2=−2<0 扔左；−4+1+2=−1<0 扔左；指针相遇 | 无解 |
| 1 | −1 | left=2,right=5：−1−1+2=0 命中，收 [-1,-1,2]；跳位后 left=3,right=4：−1+0+1=0 命中，收 [-1,0,1]；随后指针交叉 | 两组解 |
| 2 | −1 | 与 a[1] 相同，整轮跳过 | — |
| 3 | 0 | left=4,right=5：0+1+2=3>0 扔右；指针相遇 | 无解 |

最终答案 `[[-1,-1,2],[-1,0,1]]`。注意 i=2 那轮如果没有跳过，固定值又是 −1、会重新找到 [-1,0,1]，答案就重复了——外层去重跳过的必要性一眼可见。

**`max_area` 的思路：**

- **Step 1：两端就位，best 记录迄今最大面积。**
- **Step 2：算当前面积并更新 best。** 面积 = (right−left) × min(两壁高度)。
- **Step 3：扔短板。** 哪边矮就移动哪边（相等时任移一边）。
- **Step 4：指针相遇，best 即答案。**

**为什么扔短板是安全的？** 设当前左壁较矮。左壁与其他任何"更靠里"的壁配对时：宽度比现在小（距离缩短了）；高度由配对中较矮者决定，而左壁已经是不高于当前短板的那根，新配对的高度至多还是左壁的高度、不可能更高（若对面更矮则更糟）。宽变小、高不增，面积必然不超过刚检查过的这个——所以左壁的所有备选配对都可以不看。长的那根留着，因为它换一个更矮的对面时还有机会靠"自己够高"撑住面积。

**手算演示二：`max_area([1,8,6,2,5,4,8,3,7])`（“运行示例”部分 表格完整重排）**

| left | right | 短板 | 候选面积 | 动作 |
|---|---|---|---|---|
| 0 (h=1) | 8 (h=7) | 1 | 8×1=8 | 左矮，左进 |
| 1 (h=8) | 8 (h=7) | 7 | **7×7=49** | 右矮，右退 |
| 1 (h=8) | 7 (h=3) | 3 | 6×3=18 | 右退 |
| 1 (h=8) | 6 (h=8) | 8 | 5×8=40 | 相等，左进 |
| 2 (h=6) | 6 (h=8) | 6 | 4×6=24 | 左进 |
| 3 (h=2) | 6 (h=8) | 2 | 3×2=6 | 左进 |
| 4 (h=5) | 6 (h=8) | 5 | 2×5=10 | 左进 |
| 5 (h=4) | 6 (h=8) | 4 | 1×4=4 | 左进，相遇收场 |

八轮里最大的是 49，函数返回 49。“运行示例”部分 的表格正是这张轨迹——你核对一下每一行的"短板"列和"候选面积"列就能确认。

## 4. 核心代码：three_sum

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def three_sum(nums):
    a = sorted(nums)
    answer = []
    for i in range(len(a) - 2):
        if i and a[i] == a[i - 1]:
            continue
        if a[i] > 0:
            break
        left, right = (i + 1, len(a) - 1)
        while left < right:
            total = a[i] + a[left] + a[right]
            if total < 0:
                left += 1
            elif total > 0:
                right -= 1
            else:
                answer.append([a[i], a[left], a[right]])
                left += 1
                right -= 1
                while left < right and a[left] == a[left - 1]:
                    left += 1
                while left < right and a[right] == a[right + 1]:
                    right -= 1
    return answer
```

### 逐段读懂代码

### 函数 1：`three_sum`（三数和）

先贴 notebook “分段实现”部分 里的真实代码：

（对应程序见下方分段实现与完整代码。）

- **第 2 行 `a=sorted(nums)`**：排好序存进新列表 a，原数组不动。为什么必须排序：去重跳过依赖"相同值相邻"，内层消元依赖"子数组有序"。
- **第 3 行 `for i in range(len(a)-2)`**：i 是被固定的第一位，最多走到 n−3，给它右边至少留两个位置。
- **第 4 行 `if i and a[i]==a[i-1]: continue`**：外层去重。`i` 当布尔用（i=0 时为假），意思是"不是第一个元素且与前一个固定值相同就整轮跳过"。不跳的话同一个固定值会把同样的解再收一遍。
- **第 5 行 `if a[i]>0: break`**：提前终止。排序后 a[i] 是剩余最小的数，它都大于 0 说明后面三个正数凑不出 0，直接收工。这一行是优化不是正确性必需。
- **第 6 行 `left,right=i+1,len(a)-1`**：内层双指针只扫 a[i] 右边——固定值从左往右取，只往右找搭配可以保证每个三元组按"固定值≤左值≤右值"的唯一方式被发现，天然避免"同一组三个数以不同顺序被收三次"。
- **第 8–10 行**：N025 的消元三分支照搬——和小于 0（目标其实是 −a[i]，代码把 a[i] 一起加进 total 再比 0）扔左端，大于 0 扔右端。
- **第 11 行 `answer.append([a[i],a[left],a[right]])`**：命中一组解，收进答案。三个值按升序取出（有序数组的天然顺序）。
- **第 12 行 `left+=1; right-=1`**：命中后两端同时收一格——当前这对 left/right 已经用过，同样的组合不可能再成立（和固定值一起恰好等于 0，换成同值对会重复）。
- **第 13–14 行两个内层 while**：内层去重。left 跨过"和前一个位置相同"的值（`a[left]==a[left-1]`），right 对称地跨过（`a[right]==a[right+1]`）。没有这两行，`[0,0,0,0]` 会在 i=0 时反复命中 [0,0,0] 收四次。条件里 `left<right` 防止跨过头。
- **第 16 行 `return answer`**：答案按固定值升序、每组内部也升序，方便测试直接比对。

### 函数 2：`max_area`（盛水容器）

```python
def max_area(height):
    left, right = (0, len(height) - 1)
    best = 0
    while left < right:
        best = max(best, (right - left) * min(height[left], height[right]))
        if height[left] <= height[right]:
            left += 1
        else:
            right -= 1
    return best
```

- **第 2 行 `left,right=0,len(height)-1; best=0`**：两端指针加一个"迄今最优"记录器。best 初始为 0，空数组和单根数组连循环都进不去，直接返回 0。
- **第 4 行 `best=max(best,(right-left)*min(height[left],height[right]))`**：先算当前配对的面积再谈移动。宽度是 `right-left`，高度取 `min`（短板效应：装水以矮的为准），`max` 一句话完成"比现在好就更新"。
- **第 5–6 行 `if height[left]<=height[right]: left+=1 else: right-=1`**：扔短板。左不高于右就扔左（含相等的情形——相等时任扔一边都不吃亏，因为两边高度一样，谁配更近的壁都到不了更好）。注意这与三数和的移动依据不同：不看数值大小，看谁矮。
- **第 8 行 `return best`**：指针相遇时每一对"可能最优"的配对都已被算过或被上界论证排除，best 就是全局最大。

## 5. 分段实现：按顺序运行

**本章接口：** `three_sum(nums)`、`max_area(height)`

### three_sum

In [1]:
def three_sum(nums):
    a = sorted(nums)
    answer = []
    for i in range(len(a) - 2):
        if i and a[i] == a[i - 1]:
            continue
        if a[i] > 0:
            break
        left, right = (i + 1, len(a) - 1)
        while left < right:
            total = a[i] + a[left] + a[right]
            if total < 0:
                left += 1
            elif total > 0:
                right -= 1
            else:
                answer.append([a[i], a[left], a[right]])
                left += 1
                right -= 1
                while left < right and a[left] == a[left - 1]:
                    left += 1
                while left < right and a[right] == a[right + 1]:
                    right -= 1
    return answer

### max_area

In [2]:
def max_area(height):
    left, right = (0, len(height) - 1)
    best = 0
    while left < right:
        best = max(best, (right - left) * min(height[left], height[right]))
        if height[left] <= height[right]:
            left += 1
        else:
            right -= 1
    return best

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

h=[1,8,6,2,5,4,8,3,7]; l=0; r=len(h)-1; rows=[]
while l<r:
    rows.append((l,r,min(h[l],h[r]),(r-l)*min(h[l],h[r])))
    if h[l]<=h[r]: l+=1
    else: r-=1
show_table(['left','right','短板','候选面积'],rows)

left,right,短板,候选面积
0,8,1,8
1,8,7,49
1,7,3,18
1,6,8,40
2,6,6,24
3,6,2,6
4,6,5,10
5,6,4,4


## 7. 正确性、适用条件与复杂度

三数和沿用有序消元并按固定值唯一枚举。容器丢弃的短端点与任何更近端点配对，宽度更小、高度不大于已用短板，因此不优于刚检查的容器。

**代价：** 三数和 O(n²) 时间及 O(n) 排序辅助空间，不计答案；容器 O(n) 时间、O(1) 空间。

### 展开理解

**three_sum 为什么是对的？** 分三层看。第一层，任何和为 0 的三元组（设从小到大为 x≤y≤z）一定被发现：排序后 x 在某个位置 i 被固定，此时 y、z 都在 i 右边；N025 已经验证过有序两数和不漏解（每次扔掉的端点都被证明不可能参与任何解），所以内层必然扫到 (y,z)。第二层，答案不重：同一个固定值只被枚举一次（外层跳过重复 i），同一个固定值下的同一个 (left,right) 值组合只被收一次（命中后跳过重复 left/right 值）。一个三元组由三个数值唯一确定，而算法保证"固定值 + 左值 + 右值"这个三元组合每个至多触发一次 append，所以答案里没有重复。第三层，顺序天然规范：每组解按升序收集，恰好每个数值三元组只对应一种升序写法。三层合起来：该找的都找到、找到的都不重。

**max_area 为什么是对的？** 我们要确认"最优配对"没有在扔指针时被误伤。算法每一步扔掉的是当前较矮的那根壁，理由是上界论证：这根矮壁与任何更靠里的壁配对，宽度更小、高度又不可能超过当前短板（新配对高度由较矮者决定，最多还是这根矮壁自己的高度，若对面更矮则更低），面积至多等于刚检查过的值——而这个值已经记进 best 了。换句话说，被扔掉的配对全都"不优于已经算过的某个候选"，扔掉它们不损失任何最优可能；留下的较长壁继续参与后续配对。指针相遇时，所有没被排除的配对都实际算过面积，best 必然是全局最大。

**复杂度是多少？** three_sum 是 O(n²)：外层 n 次，内层双指针线性 n 次，乘起来 n²；排序 O(n log n) 被吸收。n=1000 时约一百万步内层判断，毫秒级；对比暴力 O(n³)，n=1000 是十亿步，慢一千倍。空间上 `sorted` 开了一份 O(n) 的副本（不计答案列表），“正确性与复杂度”部分 的"O(n) 排序辅助空间"指的就是它。max_area 是 O(n) 时间、O(1) 空间：每根指针单调走一遍，全程只有三个变量。n=10⁵ 时十万次乘法比较，一瞬间完成。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言分四组看：

- `assert sorted(three_sum([-1,0,1,2,-1,-4]))==[[-1,-1,2],[-1,0,1]]`：正常值，LeetCode 15 的样例。外面再包一层 `sorted` 是把函数输出的顺序规范化后对比，测的是"解集完整且无重复"——两个 −1 的存在专门考验外层去重。
- `assert three_sum([0,0,0,0])==[[0,0,0]]`：全相同元素边界。四个 0 理论上能拼出 C(4,3)=4 组，但只能收一次，专门测外层跳过（i=1,2 的固定值 0 被跳）和内层命中后的跳位是否合力防重。
- **max_area 对拍**：`for n in range(6)` 配 `product([0,1,3],repeat=n)` 枚举长度 0–5、值取 0/1/3 的全部数组（含空数组和重复高度），参考答案 ref 用生成器暴力枚举所有下标对 i<j 算面积取最大（`default=0` 兜住 n<2 时没有配对的情形）。每组断言函数值等于暴力值，测的是消元推理在含 0 高度、等高、单调等各种形态下都不出错。
- **three_sum 对拍**：`product([-1,0,1],repeat=5)` 枚举全部 3⁵=243 个长度 5 数组，参考答案用 `combinations(a,3)` 暴力枚举所有下标三元组、留下和为 0 的、各自排序后放进**集合**去重。断言把函数输出也逐组转元组放进集合比对——集合化正是为了"只比内容不比顺序不比重复"，顺便从测试角度确认函数输出天然无重复。

全部通过后打印 `N027: 所有本章断言通过`。

In [4]:
assert sorted(three_sum([-1, 0, 1, 2, -1, -4])) == [[-1, -1, 2], [-1, 0, 1]]

assert three_sum([0, 0, 0, 0]) == [[0, 0, 0]]

from itertools import product, combinations

for n in range(6):
    for a in product([0, 1, 3], repeat=n):
        ref = max(((j - i) * min(a[i], a[j]) for i in range(n) for j in range(i + 1, n)), default=0)
        assert max_area(a) == ref

for a in product([-1, 0, 1], repeat=5):
    ref = {tuple(sorted(t)) for t in combinations(a, 3) if sum(t) == 0}
    assert set(map(tuple, three_sum(a))) == ref

print('N027: 所有本章断言通过')

N027: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 推广四数和。

**练习 2：** 为跳过重复元素给出必要位置。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（推广四数和）**：方向是再加一层循环。提示：排序后用两层 for 固定前两个数 a[i]、a[j]（j 从 i+1 起步），剩下"找两个数和为 −a[i]−a[j]"用 N025 双指针；去重要在四个位置做——i 层跳过重复固定值、j 层跳过重复固定值、内层命中后跳过重复 left/right。手算示例可以用 `[1,0,-1,0,-2,2]`（经典 4Sum 样例，答案是 [[-2,-1,1,2],[-2,0,0,2],[-1,0,0,1]]），先把排序、每层固定值、内层指针的轨迹列成表再写代码。边界：数组不足 4 个元素直接空答案、目标不是 0（4Sum 通常求和为任意 target，注意提前终止条件 a[i]*4>target 这类剪枝要单独想）。
- **练习 2（为跳过重复元素给出必要位置）**：方向是论证"三处跳过一处都不能少"。提示：逐一做删除实验——去掉外层跳过，跑 `[-1,0,1,2,-1,-4]` 看是否多收 [-1,0,1]；去掉命中后的 left 跳过，跑 `[0,0,0,0]` 或 `[-2,0,0,2,2]` 看是否多收；去掉 right 跳过同理。每次实验手算一轮"没有这行跳过时 left/right 会停在哪、会不会再次命中"，用表格展示。再回答一个进阶问题：外层跳过为什么比较的是 `a[i]==a[i-1]`（向前看）而不是"向前看第一个不相等的"？两者何时等价？

两道练习共同的主线是"降维 + 防重"：每多固定一个数就多一层循环和一处去重，想清楚这个模式，NSum 一族就都通了。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def three_sum(nums):
    a = sorted(nums)
    answer = []
    for i in range(len(a) - 2):
        if i and a[i] == a[i - 1]:
            continue
        if a[i] > 0:
            break
        left, right = (i + 1, len(a) - 1)
        while left < right:
            total = a[i] + a[left] + a[right]
            if total < 0:
                left += 1
            elif total > 0:
                right -= 1
            else:
                answer.append([a[i], a[left], a[right]])
                left += 1
                right -= 1
                while left < right and a[left] == a[left - 1]:
                    left += 1
                while left < right and a[right] == a[right + 1]:
                    right -= 1
    return answer

def max_area(height):
    left, right = (0, len(height) - 1)
    best = 0
    while left < right:
        best = max(best, (right - left) * min(height[left], height[right]))
        if height[left] <= height[right]:
            left += 1
        else:
            right -= 1
    return best

# 示例与回归测试
assert sorted(three_sum([-1, 0, 1, 2, -1, -4])) == [[-1, -1, 2], [-1, 0, 1]]

assert three_sum([0, 0, 0, 0]) == [[0, 0, 0]]

from itertools import product, combinations

for n in range(6):
    for a in product([0, 1, 3], repeat=n):
        ref = max(((j - i) * min(a[i], a[j]) for i in range(n) for j in range(i + 1, n)), default=0)
        assert max_area(a) == ref

for a in product([-1, 0, 1], repeat=5):
    ref = {tuple(sorted(t)) for t in combinations(a, 3) if sum(t) == 0}
    assert set(map(tuple, three_sum(a))) == ref

print('N027: 所有本章断言通过')

N027: 所有本章断言通过


## 11. 代表题与迁移

- **15. 3Sum（三数之和）**：这题就是 `three_sum` 本尊，练的是"排序 + 固定一个数 + 内层双指针 + 三处去重"的完整组合拳；样例里的双 −1 就是去重逻辑的试金石。
- **11. Container With Most Water（盛最多水的容器）**：这题就是 `max_area` 本尊，练的是与数值消元不同的"几何消元"——用面积上界论证扔掉短板是安全的，恰好和 15 题形成"同骨架、不同理由"的对照。

两题都标为 canonical（与本章一一对应）。迁移方向：15 推广出 18（4Sum）乃至 NSum；11 的"短板上界"思想会在接雨水（42）等问题里以别的形式再现。自测问题：max_area 里两壁等高时扔哪边都行，为什么？（两边高度相同，任一边与更近壁的配对面积上界相同，谁出局都不损最优。）

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。